In [0]:
CREATE OR REPLACE VIEW
pondolens.dev_gold.vw_ap3_quadrant_trajectories AS

WITH regional_data AS (
    SELECT
        CAST(year AS INT) AS year,
        region_code,
        region_name,

        CAST(grdp_per_capita_php AS DOUBLE)
            AS grdp_per_capita_php,

        CAST(obligations_per_resident_php AS DOUBLE)
            AS obligations_per_resident_php,

        CAST(median_grdp AS DOUBLE)
            AS median_grdp,

        CAST(median_obligations AS DOUBLE)
            AS median_obligations

    FROM pondolens.dev_gold.mv_grdp_spending_population_without_ncr

    WHERE year BETWEEN 2021 AND 2025
),

selected_regions AS (
    SELECT *
    FROM regional_data
    WHERE region_code IN (
        'CAR',   -- Q1
        'VIII',  -- Q2
        'XII',   -- Q3
        'IVA'    -- Q4
    )
),

calculated AS (
    SELECT
        *,
        ROUND(
            grdp_per_capita_php - median_grdp, 2
        ) AS grdp_median_difference,

        ROUND(
            obligations_per_resident_php
            - median_obligations, 2
        ) AS budget_median_difference

    FROM selected_regions
)

SELECT
    year,
    region_code,
    region_name,

    CASE region_code
        WHEN 'CAR' THEN 'CAR'
        WHEN 'VIII' THEN 'Eastern Visayas'
        WHEN 'XII' THEN 'SOCCSKSARGEN'
        WHEN 'IVA' THEN 'CALABARZON'
    END AS region_display,

    grdp_per_capita_php,
    obligations_per_resident_php,
    median_grdp,
    median_obligations,

    -- Median-centered X and Y coordinates
    grdp_median_difference,
    budget_median_difference,

    CASE
        WHEN grdp_median_difference >= 0
         AND budget_median_difference >= 0
            THEN 'Q1 - High GRDP, High Spending'

        WHEN grdp_median_difference < 0
         AND budget_median_difference >= 0
            THEN 'Q2 - Low GRDP, High Spending'

        WHEN grdp_median_difference < 0
         AND budget_median_difference < 0
            THEN 'Q3 - Low GRDP, Low Spending'

        ELSE 'Q4 - High GRDP, Low Spending'
    END AS quadrant,

    CONCAT(region_code, ' ', CAST(year AS STRING))
        AS point_label,

    CASE region_code
        WHEN 'CAR' THEN '#2563EB'
        WHEN 'VIII' THEN '#F59E0B'
        WHEN 'XII' THEN '#16A34A'
        WHEN 'IVA' THEN '#8B5CF6'
    END AS region_color,

    year - 2020 AS year_sequence

FROM calculated

WHERE grdp_median_difference IS NOT NULL
  AND budget_median_difference IS NOT NULL;
